In [1]:
import pandas as pd
import numpy as np
import mlflow
import dagshub
import optuna
from pathlib import Path
from sklearn.model_selection import train_test_split,cross_validate
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer,TransformedTargetRegressor
from sklearn.preprocessing import OneHotEncoder,MinMaxScaler,OrdinalEncoder,PowerTransformer
from sklearn.metrics import mean_absolute_error,r2_score
from sklearn.ensemble import RandomForestRegressor

c:\Users\avanindra Bose\Urban Eats Delivery Time Prediction\UrbanEats-Delivery-Time-Prediction\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
root_dir = Path.cwd().parent
data_dir = root_dir / 'data' / 'interim' / 'urbaneats-cleaned-dataset.csv'

In [3]:
df = pd.read_csv(data_dir)

In [4]:
df.head()

,rider_id,age,ratings,restaurant_latitude,restaurant_longitude,delivery_latitude,delivery_longitude,order_date,weather,traffic,...,city,order_day,order_month,order_day_of_week,is_weekend,order_time_hour,pickup_time_minutes,order_time_of_day,distance,distance_type
0,INDORES13DEL02,37.0,4.9,22.745049,75.892471,22.765049,75.912471,2022-03-19,sunny,high,...,INDO,19,3,Saturday,1,11.0,15.0,morning,3.025149,short
1,BANGRES18DEL02,34.0,4.5,12.913041,77.683237,13.043041,77.813237,2022-03-25,stormy,jam,...,BANG,25,3,Friday,0,19.0,5.0,evening,20.183530,very_long
2,BANGRES19DEL01,23.0,4.4,12.914264,77.678400,12.924264,77.688400,2022-03-19,sandstorms,low,...,BANG,19,3,Saturday,1,8.0,15.0,morning,1.552758,short
3,COIMBRES13DEL02,38.0,4.7,11.003669,76.976494,11.053669,77.026494,2022-04-05,sunny,medium,...,COIMB,5,4,Tuesday,0,18.0,10.0,evening,7.790401,medium
4,CHENRES12DEL01,32.0,4.6,12.972793,80.249982,13.012793,80.289982,2022-03-26,cloudy,high,...,CHEN,26,3,Saturday,1,13.0,15.0,afternoon,6.210138,medium


In [5]:
df.shape

(45502, 27)

In [6]:
df.duplicated().sum()

np.int64(0)

In [7]:
# drop columns not required for model input

columns_to_drop =  ['rider_id',
                    'restaurant_latitude',
                    'restaurant_longitude',
                    'delivery_latitude',
                    'delivery_longitude',
                    'order_date',
                    "order_time_hour",
                    "order_day",
                    "city",
                    "order_day_of_week",
                    "order_month"]

df.drop(columns=columns_to_drop, inplace=True)

df

,age,ratings,weather,traffic,vehicle_condition,type_of_order,type_of_vehicle,multiple_deliveries,festival,city_type,time_taken,is_weekend,pickup_time_minutes,order_time_of_day,distance,distance_type
0,37.0,4.9,sunny,high,2,snack,motorcycle,0.0,no,urban,24,1,15.0,morning,3.025149,short
1,34.0,4.5,stormy,jam,2,snack,scooter,1.0,no,metropolitian,33,0,5.0,evening,20.183530,very_long
2,23.0,4.4,sandstorms,low,0,drinks,motorcycle,1.0,no,urban,26,1,15.0,morning,1.552758,short
3,38.0,4.7,sunny,medium,0,buffet,motorcycle,1.0,no,metropolitian,21,0,10.0,evening,7.790401,medium
4,32.0,4.6,cloudy,high,1,snack,scooter,1.0,no,metropolitian,30,1,15.0,afternoon,6.210138,medium
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
45497,30.0,4.8,windy,high,1,meal,motorcycle,0.0,no,metropolitian,32,0,10.0,morning,1.489846,short
45498,21.0,4.6,windy,jam,0,buffet,motorcycle,1.0,no,metropolitian,36,0,15.0,evening,NaN,NaN
45499,30.0,4.9,cloudy,low,1,drinks,scooter,0.0,no,metropolitian,16,0,15.0,night,4.657195,short
45500,20.0,4.7,cloudy,high,0,snack,motorcycle,1.0,no,metropolitian,26,0,5.0,afternoon,6.232393,medium


In [8]:
# check for missing values

df.isna().sum()

age                    1854
ratings                1908
weather                 525
traffic                 510
vehicle_condition         0
type_of_order             0
type_of_vehicle           0
multiple_deliveries     993
festival                228
city_type              1198
time_taken                0
is_weekend                0
pickup_time_minutes    1640
order_time_of_day      2070
distance               3630
distance_type          3630
dtype: int64

In [9]:
dagshub.init(repo_owner='AvanindraBose', repo_name='Urban-Eats-Food-Delivery-Time-Prediction', mlflow=True)

Accessing as AvanindraBose

Initialized MLflow to track repo "AvanindraBose/Urban-Eats-Food-Delivery-Time-Prediction"

Repository AvanindraBose/Urban-Eats-Food-Delivery-Time-Prediction initialized!

In [10]:
mlflow.set_tracking_uri('https://dagshub.com/AvanindraBose/Urban-Eats-Food-Delivery-Time-Prediction.mlflow')

# Droping Missing Values and then Selecting the Best HyperParameters for Random Forest Regressor.

In [11]:
temp_df = df.copy().dropna()

In [12]:
temp_df.isna().sum()

age                    0
ratings                0
weather                0
traffic                0
vehicle_condition      0
type_of_order          0
type_of_vehicle        0
multiple_deliveries    0
festival               0
city_type              0
time_taken             0
is_weekend             0
pickup_time_minutes    0
order_time_of_day      0
distance               0
distance_type          0
dtype: int64

In [13]:
temp_df.shape

(37695, 16)

In [14]:
X = temp_df.drop(columns= ['time_taken'])
y = temp_df['time_taken']

In [15]:
X.sample(10)

,age,ratings,weather,traffic,vehicle_condition,type_of_order,type_of_vehicle,multiple_deliveries,festival,city_type,is_weekend,pickup_time_minutes,order_time_of_day,distance,distance_type
41262,36.0,5.0,cloudy,high,1,buffet,scooter,1.0,no,metropolitian,0,5.0,morning,6.054295,medium
4448,20.0,4.6,sandstorms,low,1,buffet,scooter,1.0,no,metropolitian,1,15.0,night,9.348493,medium
4962,33.0,4.7,sunny,jam,0,buffet,motorcycle,1.0,no,metropolitian,0,10.0,night,12.435689,long
739,29.0,4.8,fog,medium,0,snack,motorcycle,1.0,no,metropolitian,0,15.0,afternoon,6.232380,medium
42913,36.0,4.5,cloudy,low,1,meal,motorcycle,1.0,no,metropolitian,0,10.0,night,4.537843,short
44156,31.0,4.3,stormy,low,0,meal,motorcycle,1.0,no,metropolitian,1,10.0,morning,3.105370,short
12094,39.0,4.7,sandstorms,low,2,snack,scooter,1.0,no,metropolitian,0,10.0,morning,3.025280,short
34224,20.0,4.8,sunny,medium,0,snack,motorcycle,1.0,no,metropolitian,0,10.0,evening,20.183530,very_long
5946,22.0,4.6,stormy,low,1,meal,motorcycle,0.0,no,metropolitian,0,10.0,morning,1.552762,short
12301,26.0,4.9,stormy,jam,1,meal,scooter,0.0,no,metropolitian,1,5.0,night,4.539068,short


In [16]:
y.sample(10)

28939    26
33047    10
1256     35
4621     31
13051    27
40674    14
44300    15
14219    18
14427    17
7806     28
Name: time_taken, dtype: int64

In [17]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [18]:
print("The size of train data is",X_train.shape)
print("The shape of test data is",X_test.shape)

The size of train data is (30156, 15)
The shape of test data is (7539, 15)


In [19]:
num_cols = X_train.select_dtypes(include=np.number).columns.to_list()

In [20]:
num_cols.remove('vehicle_condition')
num_cols.remove('multiple_deliveries')

In [21]:
X_train.select_dtypes(include=object).columns.to_list()

['weather',
 'traffic',
 'type_of_order',
 'type_of_vehicle',
 'festival',
 'city_type',
 'order_time_of_day',
 'distance_type']

In [22]:
ordinal_cat_cols = ['traffic','distance_type']

nominal_cat_cols = [
    'weather',
    'type_of_order',
    'type_of_vehicle',
    'festival',
    'city_type',
    'order_time_of_day'
]

In [23]:
len(num_cols + nominal_cat_cols + ordinal_cat_cols)

13

In [24]:
# generate order for ordinal encoding

traffic_order = ["low","medium","high","jam"]

distance_type_order = ["short","medium","long","very_long"]

__Testing the Pipeline.__

In [25]:
preprocessor = ColumnTransformer(
    transformers=[
        ('numerical columns',MinMaxScaler(),num_cols),
        ('nominal categorical columns',OneHotEncoder(drop='first',handle_unknown='ignore',sparse_output=False),nominal_cat_cols),
        ('ordinal categorical columns', OrdinalEncoder(categories=[traffic_order,distance_type_order]),ordinal_cat_cols)
    ],remainder='passthrough',n_jobs=-1,force_int_remainder_cols=False,verbose_feature_names_out=False
)

In [26]:
pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", RandomForestRegressor(random_state=42))
    ])

model_pipe_tt = TransformedTargetRegressor(
        regressor=pipeline,
        transformer=PowerTransformer()
    )

In [27]:
scores = cross_validate(
            model_pipe_tt,
            X_train,
            y_train,
            cv = 5,
            scoring = {
                "mae":"neg_mean_absolute_error",
                "r2":"r2"
            },
            n_jobs = -1,
            return_train_score = True
        )

In [28]:
scores

{'fit_time': array([16.96309161, 16.73801708, 16.81642079, 16.75410151, 17.15629816]),
 'score_time': array([0.32945037, 0.31702018, 0.31505466, 0.31849194, 0.32574749]),
 'test_mae': array([-3.13546104, -3.12235661, -3.12444186, -3.11345066, -3.12149127]),
 'train_mae': array([-1.15979466, -1.16200418, -1.16248186, -1.16374641, -1.15388512]),
 'test_r2': array([0.82461643, 0.8266513 , 0.82691264, 0.82564512, 0.82780124]),
 'train_r2': array([0.97533032, 0.97527773, 0.97529923, 0.97521568, 0.97559166])}

__Conducting Hyper Parameter Tuning.__

In [32]:
def build_model(params):

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", RandomForestRegressor(**params))
    ])

    model_pipe = TransformedTargetRegressor(
        regressor=pipeline,
        transformer=PowerTransformer()
    )

    return model_pipe

In [37]:
def objective(trial):

    with mlflow.start_run(run_name=f"trial_{trial.number}",nested=True) as run:

        params = {
            "n_estimators": trial.suggest_int("n_estimators",10,500),
            "max_depth": trial.suggest_int("max_depth",1,30),
            "max_features": trial.suggest_categorical("max_features",[None,"sqrt","log2"]),
            "min_samples_split": trial.suggest_int("min_samples_split",2,10),
            "min_samples_leaf": trial.suggest_int("min_samples_leaf",1,10),
            "max_samples": trial.suggest_float("max_samples",0.5,1),
            "random_state": 42,
            "n_jobs": -1,
        }

        model = build_model(params)

        scores = cross_validate(
            model,
            X_train,
            y_train,
            cv = 5,
            scoring = {
                'mae': 'neg_mean_absolute_error',
                'r2': 'r2'
            },
            n_jobs = -1,
            return_train_score = True
        )

        train_mae = -scores["train_mae"].mean()
        val_mae = -scores["test_mae"].mean()
        val_mae_std = scores["test_mae"].std()
        train_r2 = scores["train_r2"].mean()
        val_r2 = scores["test_r2"].mean()

        mlflow.log_param("model_type","RandomForestRegressor")
        mlflow.log_param("trial_number", trial.number)
        mlflow.log_params(trial.params)

        mlflow.log_metric("train_mae_mean", train_mae)
        mlflow.log_metric("val_mae_mean", val_mae)
        mlflow.log_metric("val_mae_std", val_mae_std)
        mlflow.log_metric("train_r2_mean", train_r2)
        mlflow.log_metric("val_r2_mean", val_r2)

        for i, score in enumerate(scores["test_mae"]):
            mlflow.log_metric(f"fold_{i}_val_mae", -score)

        for i, score in enumerate(scores["test_r2"]):
            mlflow.log_metric(f"fold_{i}_val_r2", score)

        trial.set_user_attr("val_mae", val_mae)
        trial.set_user_attr("val_r2", val_r2)

        return val_mae

In [38]:
# parent run
mlflow.set_experiment("Exp 3 : RF Hp Tuning")

study = optuna.create_study(direction='minimize',study_name='RF HP Tuning')

with mlflow.start_run(run_name='RF HP Tuning') as parent_run:
    mlflow.log_param("n_trials",35)
    mlflow.log_param("cv_folds",5)
    mlflow.log_param("objective_metric","val_mae")

    study.optimize(objective,n_trials=35)

    best_trial = study.best_trial

    best_params = {
        **best_trial.params,
        "random_state": 42,
        "n_jobs": -1
    }

    mlflow.log_param("best_trial_number", best_trial.number)


    for key,value in best_trial.params.items():
        mlflow.log_param(f"best_{key}",value)
    
    mlflow.log_metric("best_cv_mae",best_trial.value)

    trials_df = study.trials_dataframe()
    trials_df.to_csv("optuna_trials.csv", index=False)
    mlflow.log_artifact("optuna_trials.csv")

    best_model_pipe = build_model(best_params)
    best_model_pipe.fit(X_train,y_train)

    y_pred_train = best_model_pipe.predict(X_train)
    y_pred_test = best_model_pipe.predict(X_test)
    
    train_mae = mean_absolute_error(y_train, y_pred_train)
    test_mae = mean_absolute_error(y_test, y_pred_test)

    train_r2 = r2_score(y_train, y_pred_train)
    test_r2 = r2_score(y_test, y_pred_test)

    mlflow.log_params(best_params)

    mlflow.log_metric("train_mae", train_mae)
    mlflow.log_metric("test_mae", test_mae)
    mlflow.log_metric("train_r2", train_r2)
    mlflow.log_metric("test_r2", test_r2)

    mlflow.sklearn.log_model(best_model_pipe, "rf_hp_tuned_model")

[I 2026-06-03 08:18:53,150] A new study created in memory with name: RF HP Tuning


🏃 View run trial_0 at: https://dagshub.com/AvanindraBose/Urban-Eats-Food-Delivery-Time-Prediction.mlflow/#/experiments/3/runs/985ed0004a1341698b131740f89085c8
🧪 View experiment at: https://dagshub.com/AvanindraBose/Urban-Eats-Food-Delivery-Time-Prediction.mlflow/#/experiments/3


[I 2026-06-03 08:19:15,413] Trial 0 finished with value: 3.4391104816948306 and parameters: {'n_estimators': 278, 'max_depth': 11, 'max_features': 'sqrt', 'min_samples_split': 9, 'min_samples_leaf': 3, 'max_samples': 0.6531389939815813}. Best is trial 0 with value: 3.4391104816948306.


🏃 View run trial_1 at: https://dagshub.com/AvanindraBose/Urban-Eats-Food-Delivery-Time-Prediction.mlflow/#/experiments/3/runs/635ab305f2024d1094cfa7a275fce64a
🧪 View experiment at: https://dagshub.com/AvanindraBose/Urban-Eats-Food-Delivery-Time-Prediction.mlflow/#/experiments/3


[I 2026-06-03 08:19:53,853] Trial 1 finished with value: 4.134875461979513 and parameters: {'n_estimators': 408, 'max_depth': 7, 'max_features': 'log2', 'min_samples_split': 10, 'min_samples_leaf': 2, 'max_samples': 0.861986751024308}. Best is trial 0 with value: 3.4391104816948306.


...

🏃 View run trial_33 at: https://dagshub.com/AvanindraBose/Urban-Eats-Food-Delivery-Time-Prediction.mlflow/#/experiments/3/runs/5c238611251147609349b8054b7e115c
🧪 View experiment at: https://dagshub.com/AvanindraBose/Urban-Eats-Food-Delivery-Time-Prediction.mlflow/#/experiments/3


[I 2026-06-03 08:42:50,473] Trial 33 finished with value: 3.1050475620955025 and parameters: {'n_estimators': 433, 'max_depth': 12, 'max_features': None, 'min_samples_split': 6, 'min_samples_leaf': 4, 'max_samples': 0.8435240104833484}. Best is trial 23 with value: 3.082297725196893.


🏃 View run trial_34 at: https://dagshub.com/AvanindraBose/Urban-Eats-Food-Delivery-Time-Prediction.mlflow/#/experiments/3/runs/219ab5c6a0d94d98995c5bd45aadd0c4
🧪 View experiment at: https://dagshub.com/AvanindraBose/Urban-Eats-Food-Delivery-Time-Prediction.mlflow/#/experiments/3


[I 2026-06-03 08:43:34,718] Trial 34 finished with value: 3.084958249176413 and parameters: {'n_estimators': 467, 'max_depth': 16, 'max_features': None, 'min_samples_split': 9, 'min_samples_leaf': 5, 'max_samples': 0.791689340829295}. Best is trial 23 with value: 3.082297725196893.
c:\Users\avanindra Bose\Urban Eats Delivery Time Prediction\UrbanEats-Delivery-Time-Prediction\.venv\Lib\site-packages\sklearn\compose\_column_transformer.py:978: FutureWarning: The parameter `force_int_remainder_cols` is deprecated and will be removed in 1.9. It has no effect. Leave it to its default value to avoid this warning.
  warnings.warn(
2026/06/03 08:43:56 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/03 08:44:03 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization

🏃 View run RF HP Tuning at: https://dagshub.com/AvanindraBose/Urban-Eats-Food-Delivery-Time-Prediction.mlflow/#/experiments/3/runs/234bf7c79bf241edaf1541f22807e711
🧪 View experiment at: https://dagshub.com/AvanindraBose/Urban-Eats-Food-Delivery-Time-Prediction.mlflow/#/experiments/3


In [39]:
optuna.visualization.plot_optimization_history(study)